In [1]:
import os
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.animation import PillowWriter

In [2]:
def parse_xml(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    tracks = []

    for track in root.findall(".//track"):
        object_id = track.attrib.get("id")
        object_label = track.attrib.get("label")

        for box in track.findall("box"):
            frame = int(box.attrib["frame"])
            xtl = float(box.attrib["xtl"])
            ytl = float(box.attrib["ytl"])
            xbr = float(box.attrib["xbr"])
            ybr = float(box.attrib["ybr"])

            x_center = (xtl + xbr) / 2
            y_center = (ytl + ybr) / 2

            tracks.append({
                "id": object_id,
                "label": object_label,
                "frame": frame,
                "x": x_center,
                "y": y_center
            })

    return tracks

In [3]:
def animate_trajectory(tracks, save_path):
    fig, ax = plt.subplots()
    xs, ys = [], []

    def update(frame_num):
        ax.clear()
        ax.invert_yaxis()     # Makes origin top-left like video
        ax.set_title(f"Trajectory Animation (frame {frame_num})")
        ax.set_xlabel("X position")
        ax.set_ylabel("Y position")

        # up to current frame
        past = [t for t in tracks if t["frame"] <= frame_num]
        xs = [p["x"] for p in past]
        ys = [p["y"] for p in past]

        ax.plot(xs, ys, linewidth=2)
        return ax

    max_frame = max([t["frame"] for t in tracks])
    ani = animation.FuncAnimation(fig, update, frames=max_frame, interval=50)

    ani.save(save_path, writer=PillowWriter(fps=20))
    plt.close(fig)

In [ ]:
ANNOT_DIR = r"E:\DATA\Annotations"
OUTPUT_DIR = r"E:\DATA\AnimatedTrajectories"
os.makedirs(OUTPUT_DIR, exist_ok=True)

xml_files = [f for f in os.listdir(ANNOT_DIR) if f.endswith(".xml")]

for xml_file in xml_files:
    xml_path = os.path.join(ANNOT_DIR, xml_file)

    print("Processing:", xml_file)
    tracks = parse_xml(xml_path)

    # Group by object ID
    object_ids = set([t["id"] for t in tracks])

    for oid in object_ids:
        filtered = [t for t in tracks if t["id"] == oid]

        gif_name = f"{xml_file.replace('.xml','')}_object_{oid}.gif"
        gif_path = os.path.join(OUTPUT_DIR, gif_name)

        animate_trajectory(filtered, gif_path)
        print(" saved:", gif_name)

print("All GIFs generated!")

Processing: NO20251108-113655-002329F.xml
